# Chapter 5 — The CPA Formulation: Scientific Figures

**Figures:**
1. CPA pressure decomposition: P_cubic + P_assoc
2. CPA class hierarchy diagram
3. CPA vs SRK compressibility factor for water
4. Pressure contribution ratio (association/total) vs temperature

In [1]:
import importlib, subprocess, sys
try:
    from neqsim_dev_setup import neqsim_init, neqsim_classes
    ns = neqsim_init(recompile=False); ns = neqsim_classes(ns)
    NEQSIM_MODE = "devtools"
except Exception:
    try: import neqsim
    except ImportError: subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "neqsim"])
    from neqsim import jneqsim
    NEQSIM_MODE = "pip"
print(f"NeqSim mode: {NEQSIM_MODE}")

NeqSim project root: C:\Users\ESOL\Documents\GitHub\neqsim


NeqSim mode: pip


In [2]:
import numpy as np, matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from pathlib import Path

PALETTE = ["#2171b5", "#e6550d", "#31a354", "#756bb1", "#e7298a", "#66a61e"]
BLUE, ORANGE, GREEN, PURPLE, PINK, LIME = PALETTE
plt.rcParams.update({"font.family": "serif", "font.size": 9, "axes.labelsize": 10, "axes.titlesize": 10, "legend.fontsize": 8, "xtick.labelsize": 8, "ytick.labelsize": 8, "xtick.direction": "in", "ytick.direction": "in", "axes.linewidth": 0.6, "lines.linewidth": 1.2, "grid.linewidth": 0.3, "grid.alpha": 0.4, "savefig.dpi": 300, "figure.dpi": 150})
FIGURES_DIR = Path("../figures"); FIGURES_DIR.mkdir(exist_ok=True)
def save(fig, name): fig.savefig(str(FIGURES_DIR / name), dpi=300, bbox_inches="tight", pad_inches=0.05); plt.close(fig); print(f"Saved: {name}")
R = 8.314

## Figure 5.1 — CPA Pressure Decomposition

Shows $P_{\rm CPA} = P_{\rm SRK} + P_{\rm assoc}$ at a subcritical isotherm for water.

In [3]:
# Analytical SRK + association pressure for water at T=400K
Tc_w, Pc_w, omega_w = 647.1, 220.64e5, 0.344
T_val = 400.0  # K

# SRK parameters
a0_srk = 0.42748 * R**2 * Tc_w**2 / Pc_w
b_srk = 0.08664 * R * Tc_w / Pc_w
m_srk = 0.48 + 1.574 * omega_w - 0.176 * omega_w**2
alpha_val = (1 + m_srk * (1 - np.sqrt(T_val / Tc_w)))**2
a_val = a0_srk * alpha_val

# CPA association params for water
eps_w = 1017.3 * R  # J/mol
beta_w = 0.0692
b_w = 1.45e-5  # m3/mol (use CPA b, close to SRK b)

V = np.linspace(1.5 * b_srk, 15 * b_srk, 500)
rho = 1.0 / V  # mol/m3

# SRK pressure
P_srk = R * T_val / (V - b_srk) - a_val / (V * (V + b_srk))

# Association pressure contribution (simplified)
eta = b_w * rho / 4.0
g_rho = 1.0 / (1.0 - 1.9 * eta)
Delta = g_rho * (np.exp(eps_w / (R * T_val)) - 1.0) * b_w * beta_w
rd = rho * Delta
XA = (-1.0 + np.sqrt(1.0 + 8.0 * rd)) / (4.0 * rd)

# dP_assoc/dV = -R*T * rho^2 * sum(1/XA * dXA/drho)
# Approximate P_assoc from Helmholtz derivative
f_site = np.log(np.maximum(XA, 1e-15)) - XA / 2.0 + 0.5
A_assoc = 4.0 * f_site  # per molecule
# P_assoc \approx -rho * R * T * d(A_assoc/nRT)/d(1/rho)  (numerical differentiation)
dA_drho = np.gradient(A_assoc, rho)
P_assoc = -rho**2 * R * T_val * dA_drho

P_cpa = P_srk + P_assoc

fig, ax = plt.subplots(figsize=(3.5, 3.0))
ax.plot(V * 1e6, P_srk / 1e5, color=BLUE, linewidth=1.2, linestyle="--", label=r"$P_{\rm SRK}$")
ax.plot(V * 1e6, P_assoc / 1e5, color=ORANGE, linewidth=1.2, linestyle=":", label=r"$P_{\rm assoc}$")
ax.plot(V * 1e6, P_cpa / 1e5, color=GREEN, linewidth=1.8, label=r"$P_{\rm CPA}$")
ax.axhline(y=0, color="grey", linestyle="-", alpha=0.3, linewidth=0.5)
ax.set_xlabel(r"Molar volume (cm$^3$/mol)")
ax.set_ylabel("Pressure (bar)")
ax.set_title(f"CPA pressure decomposition (T = {T_val:.0f} K)")
ax.set_ylim(-100, 400)
ax.legend(frameon=True, fontsize=7)
ax.grid(True, alpha=0.3)
fig.tight_layout()
save(fig, "fig_ch05_01_cpa_pressure_decomposition.png")

Saved: fig_ch05_01_cpa_pressure_decomposition.png


## Figure 5.2 — CPA Compressibility Factor vs SRK (NeqSim)

In [4]:
if NEQSIM_MODE == "pip":
    SystemSrkCPAstatoil = jneqsim.thermo.system.SystemSrkCPAstatoil
    SystemSrkEos = jneqsim.thermo.system.SystemSrkEos
    ThermodynamicOperations = jneqsim.thermodynamicoperations.ThermodynamicOperations
else:
    from neqsim import jneqsim
    SystemSrkCPAstatoil = jneqsim.thermo.system.SystemSrkCPAstatoil
    SystemSrkEos = jneqsim.thermo.system.SystemSrkEos
    ThermodynamicOperations = jneqsim.thermodynamicoperations.ThermodynamicOperations

pressures = np.arange(1, 505, 20)
Z_cpa, Z_srk = [], []

for P in pressures:
    for name, cls, mr, Z_list in [("CPA", SystemSrkCPAstatoil, 10, Z_cpa),
                                   ("SRK", SystemSrkEos, "classic", Z_srk)]:
        try:
            f = cls(273.15 + 200.0, float(P))
            f.addComponent("water", 1.0)
            if isinstance(mr, int): f.setMixingRule(mr)
            else: f.setMixingRule(mr)
            ops = ThermodynamicOperations(f)
            ops.TPflash()
            f.initProperties()
            z_val = float(f.getPhase(0).getZ())
            Z_list.append(z_val)
        except Exception:
            Z_list.append(np.nan)

fig, ax = plt.subplots(figsize=(3.5, 2.8))
ax.plot(pressures, Z_cpa, color=BLUE, linewidth=1.5, label="CPA")
ax.plot(pressures, Z_srk, color=ORANGE, linewidth=1.5, linestyle="--", label="SRK")
ax.axhline(y=1.0, color="grey", linestyle=":", alpha=0.5)
ax.set_xlabel("Pressure (bar)")
ax.set_ylabel("Compressibility factor $Z$")
ax.set_title("Water $Z$-factor at 200\u00b0C")
ax.legend(frameon=True); ax.grid(True, alpha=0.3)
fig.tight_layout()
save(fig, "fig_ch05_02_compressibility_cpa_vs_srk.png")

Saved: fig_ch05_02_compressibility_cpa_vs_srk.png


## Figure 5.3 — Simplified vs Original CPA

Conceptual diagram showing the simplifications in the s-CPA model
(Kontogeorgis et al., 2006).

In [5]:
fig, ax = plt.subplots(figsize=(5.5, 2.5))

# Original CPA features vs s-CPA simplifications
features = ["RDF $g(\\rho)$", r"Association $\Delta$", "Mixing rules", "Parameters"]
original = ["Carnahan\u2013Starling\n$(1-\\eta/2)/(1-\\eta)^3$",
            "Full expression\nwith $g(\\rho)$ coupling",
            "CR-1 (general)",
            "5 per component"]
simplified = ["Simplified\n$1/(1-1.9\\eta)$",
              "Decoupled\nfrom cubic volume",
              "ECR (Elliott)",
              "5 per component"]

y_positions = np.arange(len(features))
ax.barh(y_positions + 0.15, [0.45]*4, height=0.25, color=BLUE, alpha=0.8, label="Original CPA")
ax.barh(y_positions - 0.15, [0.45]*4, height=0.25, color=ORANGE, alpha=0.8, label="s-CPA")

ax.set_yticks(y_positions)
ax.set_yticklabels(features, fontsize=8)
ax.set_xticks([])
ax.set_title("Original CPA vs Simplified CPA (s-CPA)")
ax.legend(loc="lower right", frameon=True, fontsize=7)
ax.set_xlim(0, 0.6)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.spines["bottom"].set_visible(False)

fig.tight_layout()
save(fig, "fig_ch05_03_original_vs_scpa.png")

Saved: fig_ch05_03_original_vs_scpa.png
